[Reference](https://medium.com/dsc-vit-bhopal/from-vulnerable-to-vigilant-how-adversarial-training-strengthens-gpt-and-other-large-language-a1c99b68c744)

In [ ]:
# pip install transformers

In [1]:
from transformers import pipeline

model_checkpoint = "huggingface-course/bert-finetuned-squad"
question_answerer = pipeline("question-answering", model=model_checkpoint) # Load the model

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/431M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


In [2]:
context = """
There are several places in the world where one can travel. One of them is the capital of France, the
City of Love, is Paris. On the other hand, imagine going to the place for which it is known as the
Eternal City and that is Rome, the capital of Italy.
"""

question = "The capital of France is"
adv_question = "The kapital ov Fr4nce iz"

ans = question_answerer(question=question, context=context)
adv_ans = question_answerer(question=adv_question, context=context)

In [3]:
print(f'Answer: {ans.get("answer")} ({round(ans.get("score") * 100, 2)}%)')
print(f'Adversarial Answer: {adv_ans.get("answer")} ({round(adv_ans.get("score") * 100, 2)}%)')

Answer: Paris (50.25%)
Adversarial Answer: Paris (3.7%)


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("stevhliu/my_awesome_model") # Load pre-trained Tokenizer
model = AutoModelForSequenceClassification.from_pretrained("stevhliu/my_awesome_model") # Load fine-tuned model
model.config.id2label = {0: 'Negative', 1: 'Positive'} # Change output labels
model.eval() # Set mode to evaluation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device) # Use either GPU or CPU

tokenizer_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [7]:
sample = "This movie was surprisingly good and had great acting!"

In [9]:
sample_inputs = tokenizer(sample, return_tensors="pt", truncation=True, max_length=128).to(device) # Tokenize
sample_input_ids = sample_inputs["input_ids"] # Token IDs
sample_attention_mask = sample_inputs["attention_mask"] # Attention mask (1 for real tokens, 0 for padding)

sample_embeddings = model.get_input_embeddings()(sample_input_ids) # Embeddings
sample_embeddings.retain_grad() # Retain gradient during loss backpropagation

In [10]:
outputs = model(inputs_embeds=sample_embeddings, attention_mask=sample_attention_mask, labels=torch.tensor([1]).to(device))
loss = outputs.loss
loss.backward()

In [12]:
epsilon = 0.1
grad = sample_embeddings.grad.detach()
sample_perturbed_embeddings = sample_embeddings + epsilon * grad.sign()

In [13]:
import torch.nn.functional as F

embedding_matrix = model.get_input_embeddings().weight.detach() # Get embedding matrix without gradient updates
normalized_matrix = F.normalize(embedding_matrix, p=2, dim=1) # L2 normalization of every token

def nearest_token(embedding):
    embedding = F.normalize(embedding, p=2, dim=-1) # L2 normalization of passed embedding
    similarities = torch.matmul(normalized_matrix, embedding.T) # Cosine similarity
    top_ids = similarities.argmax(dim=0) # Retrieve the most similar IDs
    return top_ids

nearest_ids = nearest_token(sample_perturbed_embeddings.squeeze(0)) # Gets IDs of most similar perturbed IDs
perturbed_tokens = tokenizer.convert_ids_to_tokens(nearest_ids) # Convert perturbed IDs to tokens
original_tokens = tokenizer.convert_ids_to_tokens(sample_input_ids.squeeze(0)) # Convert original sample IDs to tokens

In [14]:
for orig, pert in zip(original_tokens, perturbed_tokens):
    print(f"{orig:15} -> {pert}")

[CLS]           -> [CLS]
this            -> this
movie           -> movie
was             -> was
surprisingly    -> surprisingly
good            -> good
and             -> and
had             -> had
great           -> great
acting          -> acting
!               -> !
[SEP]           -> [SEP]


In [16]:
def prediction(embedding):
    with torch.no_grad():
        output = model(inputs_embeds=embedding, attention_mask=sample_attention_mask)
        pred = torch.argmax(output.logits, dim=1).item()
        return model.config.id2label[pred]

In [17]:
original_pred = prediction(sample_embeddings) # Original prediction
adversarial_pred = prediction(sample_perturbed_embeddings) # Adversarial prediction

In [18]:
print("Original Embeddings")
print(sample_embeddings)
print()

print("Perturbed Embeddings")
print(sample_perturbed_embeddings)
print()

print("Loss gradient")
print(grad)
print()

print("Loss gradient sign")
print(grad.sign())
print()

print("Original vs FGSM Perturbed Tokens")
for orig, pert in zip(original_tokens, perturbed_tokens):
    print(f"{orig:15} -> {pert}")
print()

print("Pre-Training")
print("Original prediction:")
original_pred = prediction(sample_embeddings)
print(original_pred)
print()

print("Adversarial prediction:")
adversarial_pred = prediction(sample_perturbed_embeddings)
print(adversarial_pred)
print()

Original Embeddings
tensor([[[ 0.0407, -0.0142, -0.0190,  ...,  0.0621,  0.0214,  0.0239],
         [-0.0541,  0.0139,  0.0038,  ..., -0.0125, -0.0276,  0.0150],
         [-0.0125, -0.0161, -0.0188,  ..., -0.0295,  0.0066, -0.0579],
         ...,
         [-0.0245, -0.0397, -0.0249,  ..., -0.0645,  0.0278, -0.0150],
         [ 0.0294, -0.0239, -0.0469,  ..., -0.0121,  0.0044,  0.0039],
         [-0.0181, -0.0103, -0.0098,  ..., -0.0235,  0.0053, -0.0081]]],
       device='cuda:0', grad_fn=<EmbeddingBackward0>)

Perturbed Embeddings
tensor([[[-0.0593,  0.0858, -0.1190,  ..., -0.0379,  0.1214, -0.0761],
         [-0.1541,  0.1139, -0.0962,  ..., -0.1125,  0.0724, -0.0850],
         [-0.1125,  0.0839, -0.1188,  ...,  0.0705,  0.1066, -0.1579],
         ...,
         [ 0.0755, -0.1397, -0.1249,  ...,  0.0355, -0.0722, -0.1150],
         [ 0.1294,  0.0761, -0.1469,  ...,  0.0879,  0.1044, -0.0961],
         [-0.1181,  0.0897, -0.1098,  ...,  0.0765,  0.1053,  0.0919]]],
       device='cuda: